## Desafio 3 da Trilha de Visão Computacional

Nesse desafio você irá implementar uma solução simples para realizar o rastreamento de objetos.

Para isso você deverá seguir os pasos desse documento atentamente.

### Instalação e Importação da Biblioteca

In [ ]:
!pip install ultralytics opencv-python torch yt-dlp tqdm

In [1]:
from ultralytics import YOLO
import cv2
import torch
import yt_dlp
from tqdm import tqdm

### Obtenção do vídeo

In [2]:
# gols da partida: Liverpool 7x0 Manchester United

url = "https://www.youtube.com/watch?v=iBuTEywEQ6U"

ydl_opts = {
    'outtmpl': 'video.mp4',
    'format': 'mp4',
}

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    ydl.download([url])

[youtube] Extracting URL: https://www.youtube.com/watch?v=iBuTEywEQ6U
[youtube] iBuTEywEQ6U: Downloading webpage


[youtube] iBuTEywEQ6U: Downloading android vr player API JSON
[info] iBuTEywEQ6U: Downloading 1 format(s): 18
[download] Destination: video.mp4
[download] 100% of   11.60MiB in 00:00:00 at 13.47MiB/s    


#### Info

In [3]:
cap = cv2.VideoCapture('video.mp4')

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"FPS: {fps}, Resolução: {width}x{height}, Frames: {total_frames}")

cap.release()

FPS: 25.0, Resolução: 640x360, Frames: 3345


### Processar dados

In [4]:
# Baixa o modelo
model = YOLO('yolo11n.pt')

# Jogao o modelo para o dispositivo de computação
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# processa o vídeo
results = model.track(source="video.mp4", conf=0.1, iou=0.7, verbose=False)

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs



### Salvar Vídeo

**Formato mp4**

In [6]:
fourcc = cv2.VideoWriter_fourcc(*'mp4v')

writer = cv2.VideoWriter(
    'resultado.mp4',
    fourcc,
    25.0,         
    (640, 360)   
)

for r in tqdm(results, desc="Processando o vídeo"):
    frame = r.plot(labels=False, line_width=1, font_size=0.4)
    writer.write(frame)

writer.release()
print("Vídeo salvo com sucesso!")

Processando o vídeo:   0%|          | 0/3345 [00:00<?, ?it/s]

Processando o vídeo: 100%|██████████| 3345/3345 [00:08<00:00, 395.70it/s]

Vídeo salvo com sucesso!


### Referência

- https://docs.ultralytics.com/modes/track/